# §30 — Hafıza Organı Yetenek Testi (3 koşul × 5 sonda)

"Hafıza yardımcı-işlemcisi" konumlandırmasının altındaki **ölçülmemiş** varsayımı
test eder: bilgi, KV-cache **olmadan**, O(1) durumundan *sorguyla* çıkarılabiliyor mu?

**Koşullar**
- **A — üst sınır:** tam bağlam + full attention (graft `teacher` modda). Base
  modelin her şeyi görseydi yapabileceği. Sondanın *modelden* mi *hafızadan* mı
  kaynaklandığını ayırt etmek için şart.
- **B — §22 ayarı:** hibrit + kalıcı KV-cache (eski needle testi böyleydi).
- **C — GERÇEK TEST:** hibrit + **her chunk'ta taze KV-cache**, HFP durumu taşınır.
  Mevcut chunk'tan eskisi **yalnızca** O(1) durumundan gelebilir. (Eğitim protokolü
  de böyleydi — S2 cross-chunk recall cache'siz eğitildi.)

**Sondalar:** P1 birebir (kontrol) · P2 sözcük varyantı · P3 güncellenen olgu
(sonraki değer kazanmalı) · P4 çoklu-olgu ayrımı · P5 negatif kontrol (hiç
saklanmamış anahtar — uydurmamalı).

Ön-kayıtlı kriterler RESULTS §30'da. **Eğitilmiş 6-katman checkpoint gerekir.**


In [ ]:
# --- 1. KURULUM ---
import os, subprocess, sys, glob, json, random
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
os.environ['HF_HUB_DISABLE_XET']='1'; os.environ['HF_HUB_ENABLE_HF_TRANSFER']='0'
import torch
assert torch.cuda.is_available(), 'GPU sec (T4+).'
DEV='cuda'
BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else ('/content' if os.path.exists('/content') else '.')
REPO = os.path.join(BASE,'HFP')
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/kayra-hn/HFP.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-U','transformers>=4.46','accelerate'],check=True)
os.chdir(REPO); sys.path.insert(0,REPO)
CKPT_DIR=BASE
try:
    from google.colab import drive; drive.mount('/content/drive'); CKPT_DIR='/content/drive/MyDrive/hfp_graft_ckpt'
except Exception as e: print('Drive yok:',type(e).__name__)
# Gerekirse: os.environ['HF_TOKEN']='hf_...'
print('hazir')

In [ ]:
# --- 2. MODEL + EGITILMIS 6-KATMAN CHECKPOINT ---
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache
from hfp.models.grafting import (GraftConfig, graft_llama, set_graft_mode,
                                 enable_streaming, reset_streaming, HFPGraftAttention)
MODEL_ID='Qwen/Qwen2.5-1.5B'; GRAFT_LAYERS=[3,7,11,15,19,23]
ROOTS=['/kaggle/input',CKPT_DIR,BASE,'/content/drive/MyDrive','/content']
def _find(pats):
    for r in ROOTS:
        if not r or not os.path.isdir(r): continue
        for p in pats:
            h=sorted(glob.glob(f'{r}/**/{p}',recursive=True))
            if h: return h[-1]
    return None
_cfgj=_find(['config.json']); src=os.path.dirname(_cfgj) if _cfgj and glob.glob(f'{os.path.dirname(_cfgj)}/*.safetensors') else None
if not src:
    _tk=os.environ.get('HF_TOKEN')
    try:
        from google.colab import userdata; _tk=_tk or userdata.get('HF_TOKEN')
    except Exception: pass
    assert _tk,'Model yok + HF_TOKEN yok.'
    from huggingface_hub import snapshot_download
    src=f'{BASE}/qwen_model'
    snapshot_download(repo_id=MODEL_ID,local_dir=src,token=_tk,
        allow_patterns=['*.json','*.txt','tokenizer.model','*.safetensors','*.safetensors.index.json'],max_workers=2)
tok=AutoTokenizer.from_pretrained(src)
model=AutoModelForCausalLM.from_pretrained(src,torch_dtype=torch.float32).to(DEV).eval()
graft_llama(model, GraftConfig(decay_mode='exp',write_rule='hybrid',key_feature_map='dpfp',rec_block=16), layers=GRAFT_LAYERS)
for m_ in model.modules():
    if isinstance(m_,HFPGraftAttention): m_.out_gain.data.fill_(0.1)
CKPT=_find(['hfp_graft_exp_g6*_final.pt','hfp_graft_exp_g6*_son.pt','hfp_graft*g6*final*.pt'])
assert CKPT, ('EGITILMIS 6-KATMAN CHECKPOINT YOK (hfp_graft_exp_g6*_final.pt).\n'
              '  Once ana notebook\'u GRAFT_N=6, GRAFT_FROM_MAP=None ile kos (~1-2s)\n'
              '  ve ciktiyi Drive/hfp_graft_ckpt altina koy.')
sd=torch.load(CKPT,map_location=DEV); model.load_state_dict(sd,strict=False)
set_graft_mode(model,'student'); model.config.use_cache=True
print('checkpoint:',CKPT)

In [ ]:
# --- 3. AKIS (sentetik ajan izi) + 5 SONDA ---
random.seed(0)
CHUNK=512; TRIALS=int(os.environ.get('CAP_TRIALS','8'))
FILLER=' The routine check completed and nothing of note happened during this step.'
KEYS=['deployment port','backup schedule','license server','cache directory','build target']
VALS=['9090','0300','tunis','ravenwood','falconpath']
OLD ='8080'

def build_stream(total_tokens=4096, seed=0):
    """P1..P5 icin tek akis: guncellenen olgu + 3 ek olgu; bir anahtar HIC yazilmaz."""
    r=random.Random(seed)
    k_upd, k_a, k_b, k_none = 'deployment port','license server','cache directory','backup schedule'
    v_upd, v_a, v_b = r.choice(['9090','7070','6060']), r.choice(['tunis','oslo','lima']), r.choice(['ravenwood','stonepark','elmgate'])
    facts=[(f' The {k_upd} is {OLD}. ', 0.10),          # eski deger (once)
           (f' The {k_a} is {v_a}. ',   0.30),
           (f' The {k_b} is {v_b}. ',   0.50),
           (f' The {k_upd} is {v_upd}. ',0.70)]         # GUNCEL deger (sonra)
    fid=tok(FILLER,add_special_tokens=False).input_ids
    body=[]
    while len(body)<total_tokens: body.extend(fid)
    body=body[:total_tokens]
    for txt,pos in sorted(facts,key=lambda z:-z[1]):
        ins=int(pos*len(body)); f=tok(txt,add_special_tokens=False).input_ids
        body=body[:ins]+f+body[ins:]
    probes={
      'P1_verbatim'   : (f' The {k_a} is', v_a),
      'P2_variant'    : (f' The {k_b} directory is' if 'directory' in k_b else f' The value of the {k_b} is', v_b),
      'P3_updated'    : (f' The {k_upd} is', v_upd),
      'P4_multifact'  : (f' The {k_b} is', v_b),
      'P5_negative'   : (f' The {k_none} is', None),
    }
    return body, probes, {'all_vals':[OLD,v_upd,v_a,v_b]}

@torch.no_grad()
def run(cond, body, probe_txt, gen=6):
    ids=torch.tensor(body).unsqueeze(0).to(DEV)
    q=torch.tensor(tok(probe_txt,add_special_tokens=False).input_ids).unsqueeze(0).to(DEV)
    if cond=='A':                                   # ust sinir: tam baglam + full attention
        set_graft_mode(model,'teacher'); enable_streaming(model,False)
        cache=DynamicCache(); out=model(torch.cat([ids,q],1),past_key_values=cache,use_cache=True)
    else:
        set_graft_mode(model,'student'); enable_streaming(model,True); reset_streaming(model)
        if cond=='B':                               # kalici KV-cache (§22 ayari)
            cache=DynamicCache()
            for s in range(0,ids.size(1),CHUNK):
                out=model(ids[:,s:s+CHUNK],past_key_values=cache,use_cache=True); cache=out.past_key_values
            out=model(q,past_key_values=cache,use_cache=True); cache=out.past_key_values
        else:                                       # C: her chunk TAZE cache -> eski bilgi yalniz O(1)'de
            for s in range(0,ids.size(1),CHUNK):
                model(ids[:,s:s+CHUNK],past_key_values=DynamicCache(),use_cache=True)
            cache=DynamicCache(); out=model(q,past_key_values=cache,use_cache=True); cache=out.past_key_values
    g=[]; last=out.logits[:,-1:].argmax(-1)
    for _ in range(gen):
        g.append(last.item())
        out=model(last,past_key_values=cache,use_cache=True); cache=out.past_key_values
        last=out.logits[:,-1:].argmax(-1)
    if cond!='A': enable_streaming(model,False)
    return tok.decode(g).strip()
print(f'hazir | TRIALS={TRIALS} chunk={CHUNK}')

In [ ]:
# --- 4. KOSU ---
import collections
res=collections.defaultdict(lambda: collections.defaultdict(list))
samples=collections.defaultdict(list)
for t in range(TRIALS):
    body,probes,meta=build_stream(4096,seed=t)
    for cond in ['A','B','C']:
        for pname,(ptxt,target) in probes.items():
            ans=run(cond,body,ptxt)
            if target is None:      # P5: uydurma yapiyor mu (saklanan DEGERLERDEN birini basiyor mu)
                hit=any(v.lower() in ans.lower() for v in meta['all_vals'])
            else:
                hit=target.lower() in ans.lower()
            res[cond][pname].append(hit)
            if t<2: samples[(cond,pname)].append((ptxt.strip(),target,ans))
    print(f'  deneme {t+1}/{TRIALS} bitti',flush=True)

print('\n=== SONUC (% ; P5 = YANLIS getirme orani, DUSUK iyi) ===')
P=['P1_verbatim','P2_variant','P3_updated','P4_multifact','P5_negative']
print(f"{'kosul':>6} " + ' '.join(f'{p:>13}' for p in P))
pct={}
for cond in ['A','B','C']:
    row=[]
    for p in P:
        v=100.0*sum(res[cond][p])/len(res[cond][p]); pct[(cond,p)]=v; row.append(f'{v:12.0f}%')
    lbl={'A':'A ust','B':'B cache','C':'C O(1)'}[cond]
    print(f'{lbl:>6} ' + ' '.join(row))
print('\nornek ciktilar (C kosulu):')
for p in P:
    for (ptxt,tgt,ans) in samples[('C',p)][:1]:
        print(f'  {p:>13}: "{ptxt}" -> "{ans}"   (hedef: {tgt})')
json.dump({f'{c}_{p}':pct[(c,p)] for c in "ABC" for p in P},
          open(os.path.join(BASE,'capability_results.json'),'w'), indent=2)

In [ ]:
# --- 5. ON-KAYITLI HUKUM (§30) ---
core=['P1_verbatim','P2_variant','P3_updated','P4_multifact']
A={p:pct[('A',p)] for p in core}; B={p:pct[('B',p)] for p in core}; C={p:pct[('C',p)] for p in core}
A5,B5,C5=pct[('A','P5_negative')],pct[('B','P5_negative')],pct[('C','P5_negative')]
model_limited=[p for p in core if A[p]<50]
if model_limited:
    print(f'MODEL-SINIRLI sondalar (ust sinir A bile <%50): {model_limited}')
    print('  -> bunlar 1.5B base modelin sinirini gosterir, hafiza basarisizligi DEGIL;')
    print('     hafiza hukmunden CIKARILIR.')
judge=[p for p in core if p not in model_limited]
print(f'\nHafiza hukmune giren sondalar: {judge}')
if not judge:
    print('HUKUM VERILEMEZ: ust sinir tum sondalarda basarisiz -> once daha buyuk base (7B) gerekir.')
else:
    okC   = all(C[p]>=60 for p in judge)
    close = all((A[p]-C[p])<=25 for p in judge)
    conf  = C5<=20
    collapse = any(C[p]<30 for p in judge if B[p]>=60)
    print(f'  C >=%60 hepsi: {okC} | A-C farki <=25p: {close} | P5 uydurma {C5:.0f}% (<=20 gerek): {conf}')
    print('\n=== HUKUM ===')
    if okC and close and conf:
        print('HAFIZA ORGANI DOGRULANDI: O(1) durumu, KV-cache olmadan sorgulanabilir bir')
        print('  bellek. Yardimci-islemci konumlandirmasi DESTEKLENIYOR; urun isi bunun uzerine kurulabilir.')
    elif collapse:
        print('CACHE-BAGIMLI: (B) calisiyor ama (C) cokuyor -> onceki geri getirmeler buyuk olcude')
        print('  KV-cache tarafindan tasiniyormus. O(1) durumu TEK BASINA henuz hafiza organi DEGIL.')
        print('  -> Yardimci-islemci iddiasi DESTEKLENMIYOR; urun/pitch dilinde KULLANILMAMALI.')
        print('  -> Secenekler: cache-siz geri getirme icin acikca egitmek, daha buyuk state, daha buyuk base.')
    elif C5>40:
        print(f'UYDURMA BASARISIZLIGI: P5 {C5:.0f}% -> hafiza saklanmamis degerleri uyduruyor;')
        print('  P1-P4 ne olursa olsun olgusal depo olarak kullanilamaz.')
    else:
        print('KISMI: bazi kriterler saglandi. Detay tabloya bak; iddia SINIRLI bicimde ifade edilmeli.')
        print(f'  (C ortalama {sum(C[p] for p in judge)/len(judge):.0f}%, A ortalama {sum(A[p] for p in judge)/len(judge):.0f}%)')